# Stage 1: Data Lake Foundation

Project Topic: SupplyChain Manufacturing

This notebook documents the data lake foundation for the SupplyChain Manufacturing project. The goal of Stage 1 is to design the HDFS storage structure, define landing, curated, and analytics zones, load the raw datasets into the landing zone, choose appropriate file formats, set replication strategies, and document the data lake architecture.

This design is based on the PrecisionParts manufacturing scenario, where the company is experiencing rising defect rates, inventory inefficiencies, and limited visibility across production, quality, supplier, and equipment data. By organizing these datasets into a structured data lake, the pipeline creates a foundation for integrating previously disconnected data sources and supports both batch analytics and real-time monitoring in later stages.

## Data Sources

The SupplyChain Manufacturing project uses five datasets:

- production-lines.csv.gz  
- inventory-levels.csv.gz  
- equipment-sensors.csv.gz  
- quality-inspections.json.gz  
- supplier-performance.json.gz  

These are stored locally in the data/raw/ folder. They are not committed to GitHub because they are excluded by `.gitignore`.

In [2]:
import os

raw_path = "../data/raw"

files = [
    "production-lines.csv.gz",
    "inventory-levels.csv.gz",
    "equipment-sensors.csv.gz",
    "quality-inspections.json.gz",
    "supplier-performance.json.gz"
]

for file in files:
    file_path = os.path.join(raw_path, file)
    if os.path.exists(file_path):
        print(f"FOUND: {file}")
    else:
        print(f"MISSING: {file}")

FOUND: production-lines.csv.gz
FOUND: inventory-levels.csv.gz
FOUND: equipment-sensors.csv.gz
FOUND: quality-inspections.json.gz
FOUND: supplier-performance.json.gz


## Design Alignment with Business Problem

The data lake design directly supports PrecisionParts' key challenges. Separating raw, curated, and analytics data allows production, quality, supplier, and inventory datasets to be integrated for the first time. This enables cross-domain analysis, such as linking supplier performance to defect rates and connecting machine sensor data to quality outcomes.

The curated zone ensures consistent, cleaned data for reliable analysis, while the analytics zone supports reporting on defect trends, inventory imbalances, and supplier reliability. This structure is critical for identifying root causes of rising defects and optimizing inventory decisions.

## HDFS Directory Structure

The HDFS data lake is organized into three zones:

### Landing Zone
Stores raw data exactly as provided by the course repository.

### Curated Zone
Stores cleaned, standardized, deduplicated data in Parquet format.

### Analytics Zone
Stores final aggregated outputs used for analysis, reporting, and presentation insights.

In [3]:
hdfs_directories = [
    # Landing zone
    "/data/supplychain/landing/production_lines",
    "/data/supplychain/landing/inventory_levels",
    "/data/supplychain/landing/equipment_sensors",
    "/data/supplychain/landing/quality_inspections",
    "/data/supplychain/landing/supplier_performance",

    # Curated zone
    "/data/supplychain/curated/production_lines",
    "/data/supplychain/curated/inventory_levels",
    "/data/supplychain/curated/equipment_sensors",
    "/data/supplychain/curated/quality_inspections",
    "/data/supplychain/curated/supplier_performance",

    # Analytics zone
    "/data/supplychain/analytics/defect_analysis",
    "/data/supplychain/analytics/inventory_risk",
    "/data/supplychain/analytics/supplier_scorecard",
    "/data/supplychain/analytics/equipment_monitoring"
]

print("HDFS directory creation commands:\n")

for directory in hdfs_directories:
    print(f"hdfs dfs -mkdir -p {directory}")

HDFS directory creation commands:

hdfs dfs -mkdir -p /data/supplychain/landing/production_lines
hdfs dfs -mkdir -p /data/supplychain/landing/inventory_levels
hdfs dfs -mkdir -p /data/supplychain/landing/equipment_sensors
hdfs dfs -mkdir -p /data/supplychain/landing/quality_inspections
hdfs dfs -mkdir -p /data/supplychain/landing/supplier_performance
hdfs dfs -mkdir -p /data/supplychain/curated/production_lines
hdfs dfs -mkdir -p /data/supplychain/curated/inventory_levels
hdfs dfs -mkdir -p /data/supplychain/curated/equipment_sensors
hdfs dfs -mkdir -p /data/supplychain/curated/quality_inspections
hdfs dfs -mkdir -p /data/supplychain/curated/supplier_performance
hdfs dfs -mkdir -p /data/supplychain/analytics/defect_analysis
hdfs dfs -mkdir -p /data/supplychain/analytics/inventory_risk
hdfs dfs -mkdir -p /data/supplychain/analytics/supplier_scorecard
hdfs dfs -mkdir -p /data/supplychain/analytics/equipment_monitoring


## Loading Raw Data into the Landing Zone

The raw datasets are loaded into the landing zone without modification. This preserves the original source files and allows the pipeline to be reproducible.

In [4]:
hdfs_load_commands = [
    "hdfs dfs -put ../data/raw/production-lines.csv.gz /data/supplychain/landing/production_lines/",
    "hdfs dfs -put ../data/raw/inventory-levels.csv.gz /data/supplychain/landing/inventory_levels/",
    "hdfs dfs -put ../data/raw/equipment-sensors.csv.gz /data/supplychain/landing/equipment_sensors/",
    "hdfs dfs -put ../data/raw/quality-inspections.json.gz /data/supplychain/landing/quality_inspections/",
    "hdfs dfs -put ../data/raw/supplier-performance.json.gz /data/supplychain/landing/supplier_performance/"
]

print("HDFS raw data loading commands:\n")

for command in hdfs_load_commands:
    print(command)

HDFS raw data loading commands:

hdfs dfs -put ../data/raw/production-lines.csv.gz /data/supplychain/landing/production_lines/
hdfs dfs -put ../data/raw/inventory-levels.csv.gz /data/supplychain/landing/inventory_levels/
hdfs dfs -put ../data/raw/equipment-sensors.csv.gz /data/supplychain/landing/equipment_sensors/
hdfs dfs -put ../data/raw/quality-inspections.json.gz /data/supplychain/landing/quality_inspections/
hdfs dfs -put ../data/raw/supplier-performance.json.gz /data/supplychain/landing/supplier_performance/


## File Format Choices

### Landing Zone
The landing zone keeps the original compressed files:
- CSV sources remain as `.csv.gz`
- JSON sources remain as `.json.gz`

This preserves the raw source data exactly as received. Keeping the original files also supports reproducibility and makes it possible to reprocess the pipeline from the beginning.

### Curated Zone
The curated zone will use Parquet format. Parquet is columnar, compressed, and efficient for Spark processing. It is a better choice for cleaned data because Spark can read only the columns needed for each query.

### Analytics Zone
The analytics zone will also use Parquet format. Final analytics tables will be used repeatedly for reporting and visualization, so Parquet improves query performance and reduces storage size.

### Avro Decision
Avro is not selected for this project because the provided data is already in CSV and JSON, and the main workload is analytical processing with Spark. Parquet is more appropriate for column-based analytics.

### Format Summary

| Dataset | Landing Format | Curated Format | Reason |
|---|---|---|---|
| production-lines | CSV.GZ | Parquet | Raw CSV is preserved; Parquet improves Spark analytics |
| inventory-levels | CSV.GZ | Parquet | Inventory queries benefit from columnar storage |
| equipment-sensors | CSV.GZ | Parquet | Sensor data is large and frequently filtered by factory/date |
| quality-inspections | JSON.GZ | Parquet | JSON supports nested raw structure; Parquet improves cleaned analysis |
| supplier-performance | JSON.GZ | Parquet | JSON is preserved raw; Parquet supports supplier scorecard queries |

## Data Quality Considerations

The architecture is designed to support data quality validation in later stages of the pipeline. This includes checks for missing values, validating sensor readings against expected ranges, and ensuring consistency between related datasets such as production and quality inspection records. These checks help ensure that downstream analysis is based on reliable and accurate data.

## Partitioning Strategy

The curated and analytics zones will use partitioning based on common query patterns:

- Production data: partition by `factory_id` and production date
- Inventory data: partition by date or `product_id`
- Equipment sensor data: partition by `factory_id` and sensor date
- Quality inspection data: partition by `product_id`
- Supplier performance data: partition by country or `supplier_id`

This reduces the amount of data Spark needs to scan when answering business questions.

Production and equipment sensor datasets are the largest datasets in the pipeline, so partitioning them by `factory_id` and date supports efficient per factory analysis.

## Replication Factor Strategy

Because this project is being developed in a local Docker environment, the practical replication factor is 1.

In a production HDFS environment, the recommended replication factors would be:

| Zone | Replication Factor | Justification |
|---|---:|---|
| Landing Zone | 2 | Raw data should be protected, but it can also be downloaded again from the course repository |
| Curated Zone | 3 | Cleaned datasets are reused across batch, analytics, and orchestration stages |
| Analytics Zone | 2 | Analytics outputs are important but can be regenerated from curated data |
| Streaming Checkpoints | 3 | Checkpoints are very important for recovering streaming jobs without losing state |

This strategy allows for reliability, storage cost, and also recoverability.

## Data Lake Architecture Diagram

```text
Course Repository Data
        |
        v
Local data/raw/
        |
        v
HDFS Landing Zone
(raw CSV.GZ and JSON.GZ files)
        |
        v
Spark Batch Processing
(clean, standardize, validate, deduplicate)
        |
        v
HDFS Curated Zone
(clean Parquet files)
        |
        v
Spark Joins and Aggregations
        |
        v
HDFS Analytics Zone
(final Parquet outputs)
        |
        v
Reports, Slides, and Business Insights


Equipment Sensor Data
        |
        v
Kafka Topic
        |
        v
Spark Structured Streaming
        |
        v
Real-Time Equipment Alerts
        |
        v
Airflow Streaming Monitor
```

## Stage 1 Completion Checklist

- Designed HDFS landing, curated, and analytics zones
- Documented HDFS directory creation commands
- Documented commands to load raw datasets into the landing zone
- Chose file formats for landing, curated, and analytics zones
- Justified CSV, JSON, Parquet, and Avro decisions
- Defined replication factors based on data criticality and volume
- Documented the data lake architecture with a diagram